In [0]:
%sql
use catalog pl_workforce_analysis;
create schema if not exists pl_workforce_analysis.gold_dim;
use schema gold_dim;

In [0]:
company_df = spark.read.table("pl_workforce_analysis.silver.company")
employee_df = spark.read.table("pl_workforce_analysis.silver.employees")
payroll_df = spark.read.table("pl_workforce_analysis.silver.payroll")
accounts_df = spark.read.table("pl_workforce_analysis.silver.accounts")
gl_df = spark.read.table("pl_workforce_analysis.silver.general_ledgers")
department_df = spark.read.table("pl_workforce_analysis.silver.departments")

In [0]:
from pyspark.sql.functions import col, year, month

dim_company = company_df.select('company_id', 'company_name', 'industry', 'country').dropDuplicates()
dim_account = accounts_df.select('account_id','account_name', 'account_type','pnl_section').dropDuplicates()
dim_employees = employee_df.select('employee_id','department_id', 'company_id', 'position', 'is_active').dropDuplicates()
dim_department = department_df.select('department_id').dropDuplicates()
dim_date = gl_df.select(col("posting_date").alias("date")).dropDuplicates().withColumn("year", year("date")).withColumn("month", month("date"))

In [0]:
dim_company.write.mode("overwrite").saveAsTable("gold_dim.dim_company")
dim_account.write.mode("overwrite").saveAsTable("gold_dim.dim_account")
dim_date.write.mode("overwrite").saveAsTable("gold_dim.dim_date")
dim_employees.write.mode("overwrite").saveAsTable("gold_dim.dim_employee")
dim_department.write.mode("overwrite").saveAsTable("gold_dim.dim_department")

In [0]:
%sql
create schema if not exists pl_workforce_analysis.gold_fact;
use schema gold_fact;

In [0]:
from pyspark.sql.functions import col

gl_df = gl_df.withColumn("final_amount",col("credit_amount") - col("debit_amount"))
fact_finance = gl_df.select("company_id","account_id","department_id","posting_date","fiscal_year","fiscal_period","final_amount")
fact_finance.write.mode("overwrite").saveAsTable("gold_fact.fact_finance")

In [0]:
from pyspark.sql.functions import col

fact_payroll = payroll_df.select(
    "employee_id","company_id","department_id","pay_date","gross_salary","bonus","overtime_pay","commission",
"allowances","tax_deduction","social_security","health_insurance","retirement_contribution", "other_deductions","net_salary"
).withColumn(
    "total_payroll_cost",col("gross_salary") +col("bonus") + col("overtime_pay") + col("commission") + col("allowances")
)
fact_payroll.write.mode("overwrite").saveAsTable("gold_fact.fact_payroll")